# Lesson 5.3 — From Divergence to Diagnosis

Compose which-fruit (outcome gap) + direction + health signals into a localised diagnosis.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
                                          run_pipeline)
from modules.module10.twin import (DigitalTwin, GroundTruth, outcome_gap,
                                   monitor, predict, compare_futures)
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
ground = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled={v: dict(obstacle=(vxy,0.25))})
twin = DigitalTwin(ground.world)
gap = outcome_gap(twin.simulate(), ground.run())
# which fruit + direction
which = gap['harvested_only_in_sim']
direction = 'optimistic' if v in gap['harvested_only_in_sim'] else 'pessimistic'
print('diagnosis -> fruit=%s direction=%s' % (which, direction))
checks.append(which == [v])                         # localised to the victim
checks.append(direction == 'optimistic')            # predicted harvested, actually skipped
# health signals are available to characterise HOW (from the pipeline)
w = GreenhouseWorld.demo_row(n=6, seed=1)
res = run_pipeline(w, w.q)
health = res.get('monitor', res.get('health', {}))
have_health = any(k in health for k in ('peak_error','peak_effort','min_manipulability'))
print('health signals available for diagnosis:', have_health, sorted(health)[:4])
checks.append(have_health)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


diagnosis -> fruit=['F3'] direction=optimistic


health signals available for diagnosis: True ['final_error_max', 'min_manipulability', 'min_sigma_min', 'peak_effort']
3/3 checks passed.
All checks passed.
